In [12]:
import pandas as pd
import numpy as np
import re


| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | FIFA fantasy platform unique player identifier |
| `name` | str | Player display name from FIFA fantasy (knownName or firstName + lastName) |
| `squad_id` | int | FIFA fantasy internal squad/nation identifier |
| `team` | str | Nation name in English, mapped from squad_id |
| `position` | str | Fantasy position: DEF, MID, FWD, GK |
| `price` | float | Fantasy selection price in FIFA fantasy currency |
| `status` | str | Player availability: playing, transferred |
| `total_points` | int | Total fantasy points accumulated across all completed rounds |
| `avg_points` | float | Average fantasy points per round played |
| `matches_played` | int | Number of rounds with non-empty stat entries |
| `percent_selected` | float | Percentage of fantasy teams that have selected this player |
| `round_i_points` | int | Fantasy points earned in round i; one column per round played; missing if player did not participate (filled to 0 post-processing) |
| `next_fixture` | int | FIFA fantasy fixture ID for the next scheduled match |

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Player name as it appears in Betano scorer markets |
| `betano_match_score` | float | Fuzzy match confidence score 0 to 100; 100 indicates exact or manual override match |

| Column | Type | Description |
|---|---|---|
| `anytime_scorer_odd` | float | Betano decimal odd for player to score at any point; defaults to 150.0 if team has a match but player has no Betano entry |
| `anytime_scorer_prob` | float | Raw implied probability from anytime scorer odd (1 / odd) |
| `first_scorer_odd` | float | Betano decimal odd for player to score the first goal of the match |
| `first_scorer_prob` | float | Raw implied probability from first scorer odd (1 / odd) |
| `last_scorer_odd` | float | Betano decimal odd for player to score the last goal of the match |

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team name for the player's next match |
| `match_away` | str | Away team name for the player's next match |
| `is_home` | bool | True if the player's team is the home side |
| `opponent` | str | Opposing team name |
| `match_home_win_odd` | float | Betano decimal odd for home team victory |
| `match_draw_odd` | float | Betano decimal odd for a draw |
| `match_away_win_odd` | float | Betano decimal odd for away team victory |
| `match_btts_prob` | float | De-vigged probability that both teams score |
| `match_over_25_odd` | float | Betano decimal odd for over 2.5 total goals |
| `match_over_25_prob` | float | De-vigged probability of over 2.5 total goals |
| `match_over_35_odd` | float | Betano decimal odd for over 3.5 total goals |
| `team_over_05_odd` | float | Betano decimal odd for the player's team to score at least 1 goal |
| `team_over_15_odd` | float | Betano decimal odd for the player's team to score at least 2 goals |
| `team_score_prob` | float | De-vigged probability the player's team scores at least once |
| `team_score_2_prob` | float | Implied probability the player's team scores 2 or more goals (1 / team_over_15_odd) |
| `team_cs_prob` | float | De-vigged probability the player's team keeps a clean sheet |
| `team_qualify_odd` | float | Betano decimal odd for the player's team to advance to the next round |
| `team_win2_odd` | float | Betano decimal odd for the player's team to win by 2 or more goals |
| `opp_over_05_odd` | float | Betano decimal odd for the opponent to score at least 1 goal |
| `opp_score_prob` | float | De-vigged probability the opponent scores at least once |
| `opp_cs_prob` | float | De-vigged probability the opponent keeps a clean sheet |

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals scored during the tournament, cumulative across all rounds |
| `stat_AS` | int | Assists, cumulative |
| `stat_CS` | int | Clean sheets, cumulative |
| `stat_GC` | int | Goals conceded, cumulative |
| `stat_MP` | int | Minutes played, cumulative |
| `stat_YC` | int | Yellow cards, cumulative |
| `stat_RC` | int | Red cards, cumulative |
| `stat_ST` | int | Shots on target, cumulative |
| `stat_SB` | int | Shots blocked, cumulative |
| `stat_CC` | int | Chances created, cumulative |
| `stat_PS` | int | Penalties saved, cumulative (GK) |
| `stat_T` | int | Tackles, cumulative |
| `stat_S` | int | Saves, cumulative (GK) |
| `stat_SXI` | int | Times named in the starting XI, cumulative |
| `stat_OG` | int | Own goals, cumulative |
| `stat_PC` | int | Penalties committed, cumulative |
| `stat_PW` | int | Penalties won, cumulative |
| `stat_FK` | int | Free kicks earned, cumulative |

| Column | Type | Description |
|---|---|---|
| `round_i_GS` | int | Goals scored in round i; one column per round; absent (NaN) if player did not play that round |
| `round_i_AS` | int | Assists in round i |
| `round_i_CS` | int | Clean sheet in round i |
| `round_i_GC` | int | Goals conceded in round i |
| `round_i_MP` | int | Minutes played in round i |
| `round_i_YC` | int | Yellow cards in round i |
| `round_i_RC` | int | Red cards in round i |
| `round_i_ST` | int | Shots on target in round i |
| `round_i_SB` | int | Shots blocked in round i |
| `round_i_CC` | int | Chances created in round i |
| `round_i_PS` | int | Penalties saved in round i (GK) |
| `round_i_T` | int | Tackles in round i |
| `round_i_S` | int | Saves in round i (GK) |
| `round_i_SXI` | int | 1 if named in starting XI in round i, 0 otherwise |
| `round_i_OG` | int | Own goals in round i |
| `round_i_PC` | int | Penalties committed in round i |
| `round_i_PW` | int | Penalties won in round i |
| `round_i_FK` | int | Free kicks earned in round i |


| Column                         | Type  | Description                                                                                           |
| ------------------------------ | ----- | ----------------------------------------------------------------------------------------------------- |
| `started_last_match`           | int   | 1 if the player started the most recent round, 0 otherwise                                            |
| `starts_last_3`                | int   | Number of starts across the last 3 rounds                                                             |
| `starts_last_5`                | int   | Number of starts across the last 5 rounds                                                             |
| `start_rate`                   | float | Fraction of matches in which the player has started (`stat_SXI / matches_played`)                     |
| `minutes_last_3_avg`           | float | Average minutes played over the last 3 rounds                                                         |
| `minutes_last_5_avg`           | float | Average minutes played over the last 5 rounds                                                         |
| `goal_rate`                    | float | Goals scored per minute played (`stat_GS / stat_MP`)                                                  |
| `goals_per_start`              | float | Goals scored per start (`stat_GS / stat_SXI`)                                                         |
| `shots_on_target_per90`        | float | Shots on target per 90 minutes                                                                        |
| `goals_per_shot_on_target`     | float | Goals scored divided by shots on target                                                               |
| `recent_goals_last3`           | int   | Goals scored across the last 3 rounds                                                                 |
| `goal_streak`                  | int   | Number of consecutive rounds ending with the most recent in which the player scored at least one goal |
| `goal_expectation`             | float | Goal scoring proxy computed as `anytime_scorer_prob × team_score_prob`                                |
| `goal_expectation2`            | float | Goal scoring proxy computed as `anytime_scorer_prob × team_score_2_prob`                              |
| `assist_rate`                  | float | Assists per minute played (`stat_AS / stat_MP`)                                                       |
| `recent_assists_last3`         | int   | Assists recorded across the last 3 rounds                                                             |
| `chance_created_per90`         | float | Chances created per 90 minutes                                                                        |
| `recent_chances_created_per90` | float | Chances created per 90 minutes across the last 3 rounds                                               |
| `assist_expectation`           | float | Assist proxy computed as `chance_created_per90 × team_score_prob`                                     |
| `cs_rate`                      | float | Clean sheets per start (`stat_CS / stat_SXI`)                                                         |
| `recent_cs_rate`               | float | Average clean sheets across the last 3 rounds                                                         |
| `expected_minutes`             | float | Average minutes played over the last 3 rounds                                                         |
| `cs_expectation`               | float | Clean sheet proxy computed as `team_cs_prob × expected_minutes`                                       |
| `gc_per90`                     | float | Goals conceded per 90 minutes                                                                         |
| `expected_gc_penalty`          | float | Expected goals conceded penalty proxy computed as `opp_score_prob × (1 − team_cs_prob)`               |
| `tackles_per90`                | float | Tackles per 90 minutes                                                                                |
| `recent_tackles_per90`         | float | Tackles per 90 minutes across the last 3 rounds                                                       |
| `expected_tackle_points`       | float | Expected FIFA tackle points (`tackles_per90 / 3`)                                                     |
| `cc_per90`                     | float | Chances created per 90 minutes                                                                        |
| `recent_cc_per90`              | float | Chances created per 90 minutes across the last 3 rounds                                               |
| `expected_cc_points`           | float | Expected FIFA chance creation points (`cc_per90 / 2`)                                                 |
| `sot_per90`                    | float | Shots on target per 90 minutes                                                                        |
| `recent_sot_per90`             | float | Shots on target per 90 minutes across the last 3 rounds                                               |
| `expected_sot_points`          | float | Expected FIFA shot-on-target points (`sot_per90 / 2`)                                                 |
| `saves_per90`                  | float | Saves per 90 minutes                                                                                  |
| `recent_saves_per90`           | float | Saves per 90 minutes across the last 3 rounds                                                         |
| `expected_save_points`         | float | Expected FIFA save points (`saves_per90 / 3`)                                                         |
| `yc_per90`                     | float | Yellow cards per 90 minutes                                                                           |
| `rc_per90`                     | float | Red cards per 90 minutes                                                                              |
| `penalty_conceded_rate`        | float | Penalties conceded per 90 minutes                                                                     |
| `own_goal_rate`                | float | Own goals per 90 minutes                                                                              |
| `penalties_won_per90`          | float | Penalties won per 90 minutes                                                                          |
| `recent_penalties_won`         | int   | Penalties won across the last 3 rounds                                                                |
| `points_last1`                 | int   | Fantasy points scored in the most recent round                                                        |
| `points_last3_avg`             | float | Average fantasy points across the last 3 rounds                                                       |
| `points_last5_avg`             | float | Average fantasy points across the last 5 rounds                                                       |
| `weighted_points`              | float | Weighted recent fantasy points (`4×last1 + 3×last2 + 2×last3 + 1×last4`)                              |
| `rolling_std_points`           | float | Standard deviation of fantasy points across the last 5 rounds                                         |
| `rolling_max_points`           | int   | Maximum fantasy points scored in the last 5 rounds                                                    |
| `team_win_prob`                | float | Raw implied probability that the player's team wins its next match (`1 / win odd`)                    |
| `goal_involvement`             | float | Combined attacking proxy (`anytime_scorer_prob + assist_rate`)                                        |
| `attacking_match`              | float | Attacking environment proxy (`team_score_prob × match_over_25_prob`)                                  |
| `price_rank_team_position`     | float | Player price rank within players of the same team and fantasy position (1 = highest price)            |
| `points_rank_team_position`    | float | Total fantasy points rank within players of the same team and fantasy position (1 = most points)      |
| `minutes_rank_team_position`   | float | Minutes played rank within players of the same team and fantasy position (1 = most minutes)           |
| `starts_rank_team_position`    | float | Starts rank within players of the same team and fantasy position (1 = most starts)                    |
| `anytime_rank_team_position`   | float | Anytime scorer odds rank within players of the same team and fantasy position (1 = lowest odds)       |
| `shots_rank_team_position`     | float | Shots on target rank within players of the same team and fantasy position (1 = most shots on target)  |
| `cc_rank_team_position`        | float | Chances created rank within players of the same team and fantasy position (1 = most chances created)  |
| `tackles_rank_team_position`   | float | Tackles rank within players of the same team and fantasy position (1 = most tackles)                  |


| Column | Type | Description |
|---|---|---|
| `club_MP` | int | Matches played in top 5 European leagues, 2025/26 season |
| `club_Starts` | int | Matches started |
| `club_Min` | int | Total minutes played |
| `club_90s` | float | Minutes played expressed as full 90-minute equivalents |
| `club_Gls` | int | Goals scored |
| `club_Ast` | int | Assists |
| `club_G+A` | int | Goals plus assists |
| `club_G-PK` | int | Non-penalty goals |
| `club_TklW` | int | Tackles won |
| `club_Int` | int | Interceptions |
| `club_GA` | int | Goals conceded (GK only) |
| `club_Saves` | int | Saves (GK only) |
| `club_CS` | int | Clean sheets |
| `club_PKA` | int | Penalties faced (GK only) |
| `club_PKsv` | int | Penalties saved (GK only) |
| `club_CrdY` | int | Yellow cards |
| `club_CrdR` | int | Red cards |
| `club_PK` | int | Penalties scored |
| `club_PKatt` | int | Penalty attempts |
| `club_SoTA` | int | Shots on target faced (GK only) |
| `club_PKm` | int | Penalties missed |
| `club_Sh` | int | Shots |
| `club_SoT` | int | Shots on target |
| `club_Fls` | int | Fouls committed |
| `club_Fld` | int | Fouls drawn |
| `club_Off` | int | Offsides |
| `club_Crs` | int | Crosses |
| `club_OG` | int | Own goals |
| `club_2CrdY` | int | Second yellow cards |
| `club_Save%` | float | Save percentage (GK); unreliable if multi-club |
| `club_CS%` | float | Clean sheet percentage; unreliable if multi-club |
| `club_GA90` | float | Goals conceded per 90 (GK); unreliable if multi-club |
| `club_SoT%` | float | Shots on target percentage; unreliable if multi-club |
| `club_Sh/90` | float | Shots per 90; unreliable if multi-club |
| `club_SoT/90` | float | Shots on target per 90; unreliable if multi-club |
| `club_G/Sh` | float | Goals per shot; unreliable if multi-club |
| `club_G/SoT` | float | Goals per shot on target; unreliable if multi-club |
| `club_Pos` | str | Position code from FBref (FW, MF, DF, GK or combinations) |
| `club_Squad` | str | Club or clubs played for; slash-separated if multi-club |
| `club_Comp` | str | League or leagues played in |
| `club_Age` | int | Player age at time of data collection |
| `club_fbref_multi_club` | bool | True if player appeared for more than one club; rate stats unreliable when True |
| `club_matched_player` | str | Player name as matched in the FBref dataset |
| `club_match_dist` | int | Levenshtein edit distance between FIFA name and matched FBref name; 0 is exact |


In [13]:
df= pd.read_csv("fantasy_enriched.csv")

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Columns: 106 entries, fifa_id to clubs_match_dist
dtypes: bool(1), float64(88), int64(4), object(1), str(12)
memory usage: 470.1+ KB


In [15]:
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\n".join(df.columns))

Dataset shape: 520 rows x 106 columns
fifa_id
name
squad_id
team
position
price
status
total_points
avg_points
matches_played
percent_selected
round_1_points
round_2_points
round_3_points
round_4_points
next_fixture
betano_matched_name
betano_match_score
anytime_scorer_odd
anytime_scorer_prob
first_scorer_odd
first_scorer_prob
last_scorer_odd
match_home
match_away
is_home
opponent
match_home_win_odd
match_draw_odd
match_away_win_odd
match_btts_prob
match_over_25_odd
match_over_25_prob
match_over_35_odd
team_over_05_odd
team_over_15_odd
team_score_prob
team_score_2_prob
team_cs_prob
team_qualify_odd
team_win2_odd
opp_over_05_odd
opp_score_prob
opp_cs_prob
stat_GS
stat_AS
stat_CS
stat_GC
stat_MP
stat_YC
stat_RC
stat_ST
stat_SB
stat_CC
stat_PS
stat_T
stat_S
stat_SXI
stat_OG
stat_PC
stat_PW
stat_FK
club_MP
club_Starts
club_Min
club_90s
club_Gls
club_Ast
club_G+A
club_G-PK
club_TklW
club_Int
club_GA
club_Saves
club_CS
club_PKA
club_PKsv
club_CrdY
club_CrdR
club_PK
club_PKatt
club_SoTA
club_

In [16]:
print(df.shape)

(520, 106)


In [17]:
df.isnull().sum().sum()

np.int64(13624)

In [18]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

/tmp/ipykernel_5245/2784395380.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


In [19]:
# Table for missing values distribution and percentage.
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"n_missing": missing, "%_missing": missing_pct})
missing_df = missing_df[missing_df["n_missing"] > 0].sort_values("n_missing", ascending=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

missing_df

,n_missing,%_missing
club_GA90,502,96.54
club_CS%,502,96.54
club_Save%,502,96.54
round_4_points,395,75.96
club_G/SoT,292,56.15
club_SoT%,277,53.27
club_G/Sh,277,53.27
club_Int,254,48.85
club_TklW,254,48.85
club_Saves,254,48.85


In [20]:
# %% Scorer odds: missing = player has no realistic scorer market on Betano
# Covers GKs, bench warmers, players not listed because their odds would be
# astronomically high. We use 150.0 — Betano's practical ceiling for listed
# players. Treating unlisted players as strictly less likely (i.e. using 150.0)
# is a conservative underestimate of their true odds.
# Note: even the third GK is theoretically capable of scoring a penalty in
# extra time. 150.0 is not 0. It is just very, very, very close to 0 in
# probability terms (p ≈ 0.0067). This is intentional.
IMPOSSIBLE_ODD = 150.0

scorer_odd_cols = sorted([c for c in df.columns if c.endswith("_scorer_odd")])
for col in scorer_odd_cols:
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna(IMPOSSIBLE_ODD)
    prob_col = col.replace("_odd", "_prob")
    if prob_col in df.columns:
        df[prob_col] = (1 / df[col]).round(4)
    print(f"  {col}: filled {n_filled} NaN → {IMPOSSIBLE_ODD}  (prob = {round(1/IMPOSSIBLE_ODD, 4)})")


  anytime_scorer_odd: filled 0 NaN → 150.0  (prob = 0.0067)
  first_scorer_odd: filled 81 NaN → 150.0  (prob = 0.0067)
  last_scorer_odd: filled 81 NaN → 150.0  (prob = 0.0067)


In [21]:
# Table for missing values distribution and percentage.
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"n_missing": missing, "%_missing": missing_pct})
missing_df = missing_df[missing_df["n_missing"] > 0].sort_values("n_missing", ascending=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

missing_df

,n_missing,%_missing
club_Save%,502,96.54
club_GA90,502,96.54
club_CS%,502,96.54
round_4_points,395,75.96
club_G/SoT,292,56.15
club_G/Sh,277,53.27
club_SoT%,277,53.27
club_TklW,254,48.85
club_Int,254,48.85
club_Ast,254,48.85


missing values in club at 98% is anyone that is either not a goalkeeper or not in the stats sheet
club_G/SoT	358	57.28
club_SoT%	340	54.40
club_G/Sh	340	54.40 these slightly higher than the others i assume are defenders and others who have not had goals or shots or cannot have a value assingned to this metric is like 20 rows more than the common mnar
all the club 49.92 are players playing in the world cup outside top 5 leagues number will change based on left over teams ofc
round missing values are players who have not played / entered minutes. so 0, but different than a 0 from playing but not making points.
betano scorers names are goalkeepers, very unkown players players who will not even play basically goal keepers and the back of the row in teams national teams
all betano names are assinged so these are unlisted players. 
the team prob is weird betano fetcher missed some odds 

this code is held by strings from beggining to end any introdction of didferent format different national teams differnnt names itd break 

this code would obv need a lot of pre processing in the clubs variables, but they wont be used almost anywhere and are here only to add minor information to some of the secondary stats such as saves yellow cards. but unfortunately to keep everything on similar step they wont be used almost in its entirety and will be dropped if needed. thus, well focus on if any fifa fantasy / betano odds need pre processing

In [23]:
# Go through every column in the dataframe
for column in df.columns:

    # Only check columns whose name ends with "prob"
    if column.endswith("prob"):

        # Go through every value in that column
        for value in df[column]:

            # Check if the value is outside the range [0, 1]
            if value < 0 or value > 1:
                print(f"{column}: {value}")



for column in df.columns:
    if column.endswith("prob"):
        for index, value in enumerate(df[column]):
            if value < 0 or value > 1:
                print(f"Row {index}, column '{column}': {value}")

In [ ]:
def minutes_engineered(df):

    # Find all SXI columns in order
    sxi_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_SXI")],
        key=lambda x: int(x.split("_")[1])
    )

    # Find all MP columns in order
    mp_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_MP")],
        key=lambda x: int(x.split("_")[1])
    )

    # Last match
    df["started_last_match"] = df[sxi_cols[-1]]

    # Last 3 starts
    df["starts_last_3"] = df[sxi_cols[-3:]].sum(axis=1)

    # Last 5 starts
    df["starts_last_5"] = df[sxi_cols[-5:]].sum(axis=1)

    # Overall start rate
    df["start_rate"] = df["stat_SXI"] / df["matches_played"]

    # Average minutes last 3 matches
    df["minutes_last_3_avg"] = df[mp_cols[-3:]].mean(axis=1)

    # Average minutes last 5 matches
    df["minutes_last_5_avg"] = df[mp_cols[-5:]].mean(axis=1)

    return df

In [ ]:
def goals_engineered(df):

    # Find goal columns in order
    goal_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_GS")],
        key=lambda x: int(x.split("_")[1])
    )

    # Goals per minute
    df["goal_rate"] = df["stat_GS"] / df["stat_MP"]

    # Goals per start
    df["goals_per_start"] = df["stat_GS"] / df["stat_SXI"]

    # Shots on target per 90
    df["shots_on_target_per90"] = df["stat_ST"] * 90 / df["stat_MP"]

    # Goals per shot on target
    df["goals_per_shot_on_target"] = df["stat_GS"] / df["stat_ST"]

    # Goals in last 3 rounds
    df["recent_goals_last3"] = df[goal_cols[-3:]].sum(axis=1)

    # Goal streak (number of consecutive matches with a goal, starting from most recent)
    streak = []
    for _, row in df.iterrows():
        s = 0
        for col in reversed(goal_cols):
            if row[col] > 0:
                s += 1
            else:
                break
        streak.append(s)
    df["goal_streak"] = streak

    # Rank of anytime scorer odds within team and position (1 = favourite)
    df["anytime_rank_team_position"] = (
        df.groupby(["team", "position"])["anytime_scorer_odd"]
          .rank(method="dense")
    )

    # Percentile of anytime scorer odds within team and position
    df["anytime_percentile_team_position"] = (
        df.groupby(["team", "position"])["anytime_scorer_odd"]
          .rank(pct=True)
    )

    # Expected goal involvement proxies
    df["goal_expectation"] = (
        df["anytime_scorer_prob"] * df["team_score_prob"]
    )

    df["goal_expectation2"] = (
        df["anytime_scorer_prob"] * df["team_score_2_prob"]
    )

    return df

In [ ]:
def assists_engineered(df):

    # Find assist, chance created and minutes columns in order
    assist_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_AS")],
        key=lambda x: int(x.split("_")[1])
    )

    cc_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_CC")],
        key=lambda x: int(x.split("_")[1])
    )

    mp_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_MP")],
        key=lambda x: int(x.split("_")[1])
    )

    # Assists per minute
    df["assist_rate"] = df["stat_AS"] / df["stat_MP"]

    # Assists in last 3 rounds
    df["recent_assists_last3"] = df[assist_cols[-3:]].sum(axis=1)

    # Chances created per 90
    df["chance_created_per90"] = df["stat_CC"] * 90 / df["stat_MP"]

    # Chances created per 90 over the last 3 rounds
    df["recent_chances_created_per90"] = (
        df[cc_cols[-3:]].sum(axis=1) * 90 /
        df[mp_cols[-3:]].sum(axis=1)
    )

    # Assist expectation
    df["assist_expectation"] = (
        df["chance_created_per90"] * df["team_score_prob"]
    )

    return df

In [ ]:
def clean_sheet_engineered(df):

    # Find clean sheet columns in order
    cs_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_CS")],
        key=lambda x: int(x.split("_")[1])
    )

    # Clean sheet rate
    df["cs_rate"] = df["stat_CS"] / df["stat_SXI"]

    # Clean sheet rate over last 3 matches
    df["recent_cs_rate"] = df[cs_cols[-3:]].mean(axis=1)

    # Expected minutes (use last 3 matches)
    mp_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_MP")],
        key=lambda x: int(x.split("_")[1])
    )

    df["expected_minutes"] = df[mp_cols[-3:]].mean(axis=1)

    # Clean sheet expectation
    df["cs_expectation"] = (
        df["team_cs_prob"] * df["expected_minutes"]
    )

    return df


def goals_conceded_engineered(df):

    # Goals conceded per 90
    df["gc_per90"] = df["stat_GC"] * 90 / df["stat_MP"]

    # Expected goals conceded penalty
    df["expected_gc_penalty"] = (
        df["opp_score_prob"] * (1 - df["team_cs_prob"])
    )

    return df

In [ ]:
def midfield_engineered(df):

    # Find tackle, chance created and minutes columns in order
    tackle_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_T")],
        key=lambda x: int(x.split("_")[1])
    )

    cc_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_CC")],
        key=lambda x: int(x.split("_")[1])
    )

    mp_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_MP")],
        key=lambda x: int(x.split("_")[1])
    )

    # Tackles per 90
    df["tackles_per90"] = df["stat_T"] * 90 / df["stat_MP"]

    # Recent tackles per 90
    df["recent_tackles_per90"] = (
        df[tackle_cols[-3:]].sum(axis=1) * 90 /
        df[mp_cols[-3:]].sum(axis=1)
    )

    # Expected tackle points
    df["expected_tackle_points"] = df["tackles_per90"] / 3

    # Chances created per 90
    df["cc_per90"] = df["stat_CC"] * 90 / df["stat_MP"]

    # Recent chances created per 90
    df["recent_cc_per90"] = (
        df[cc_cols[-3:]].sum(axis=1) * 90 /
        df[mp_cols[-3:]].sum(axis=1)
    )

    # Expected chance creation points
    df["expected_cc_points"] = df["cc_per90"] / 2

    return df

In [ ]:
def forward_engineered(df):

    # Find shots on target and minutes columns in order
    sot_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_ST")],
        key=lambda x: int(x.split("_")[1])
    )

    mp_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_MP")],
        key=lambda x: int(x.split("_")[1])
    )

    # Shots on target per 90
    df["sot_per90"] = df["stat_ST"] * 90 / df["stat_MP"]

    # Recent shots on target per 90
    df["recent_sot_per90"] = (
        df[sot_cols[-3:]].sum(axis=1) * 90 /
        df[mp_cols[-3:]].sum(axis=1)
    )

    # Expected shot points
    df["expected_sot_points"] = df["sot_per90"] / 2

    return df

In [ ]:
def goalkeeper_engineered(df):

    # Find saves and minutes columns in order
    save_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_S")],
        key=lambda x: int(x.split("_")[1])
    )

    mp_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_MP")],
        key=lambda x: int(x.split("_")[1])
    )

    # Saves per 90
    df["saves_per90"] = df["stat_S"] * 90 / df["stat_MP"]

    # Recent saves per 90
    df["recent_saves_per90"] = (
        df[save_cols[-3:]].sum(axis=1) * 90 /
        df[mp_cols[-3:]].sum(axis=1)
    )

    # Expected save points
    df["expected_save_points"] = df["saves_per90"] / 3

    return df

In [ ]:
def discipline_engineered(df):

    # Yellow cards per 90
    df["yc_per90"] = df["stat_YC"] * 90 / df["stat_MP"]

    # Red cards per 90
    df["rc_per90"] = df["stat_RC"] * 90 / df["stat_MP"]

    # Penalties conceded per 90
    df["penalty_conceded_rate"] = df["stat_PC"] * 90 / df["stat_MP"]

    # Own goals per 90
    df["own_goal_rate"] = df["stat_OG"] * 90 / df["stat_MP"]

    return df

In [ ]:
def penalties_engineered(df):

    # Find penalty won and minutes columns in order
    pw_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_PW")],
        key=lambda x: int(x.split("_")[1])
    )

    # Penalties won per 90
    df["penalties_won_per90"] = df["stat_PW"] * 90 / df["stat_MP"]

    # Penalties won in last 3 rounds
    df["recent_penalties_won"] = df[pw_cols[-3:]].sum(axis=1)

    return df

In [ ]:
def form_engineered(df):

    # Find points columns in order
    point_cols = sorted(
        [c for c in df.columns if c.startswith("round_") and c.endswith("_points")],
        key=lambda x: int(x.split("_")[1])
    )

    # Last match points
    df["points_last1"] = df[point_cols[-1]]

    # Average points last 3 matches
    df["points_last3_avg"] = df[point_cols[-3:]].mean(axis=1)

    # Average points last 5 matches
    df["points_last5_avg"] = df[point_cols[-5:]].mean(axis=1)

    # Weighted recent points (4,3,2,1)
    df["weighted_points"] = (
        df[point_cols[-1]] * 4 +
        df[point_cols[-2]] * 3 +
        df[point_cols[-3]] * 2 +
        df[point_cols[-4]] * 1
    )

    # Standard deviation over last 5 matches
    df["rolling_std_points"] = df[point_cols[-5:]].std(axis=1)

    # Maximum points over last 5 matches
    df["rolling_max_points"] = df[point_cols[-5:]].max(axis=1)

    return df

In [ ]:
def relative_team_engineered(df):

    # Lower is better for price and odds
    df["price_rank_team_position"] = (
        df.groupby(["team", "position"])["price"]
        .rank(method="dense", ascending=False)
    )

    df["points_rank_team_position"] = (
        df.groupby(["team", "position"])["total_points"]
        .rank(method="dense", ascending=False)
    )

    df["minutes_rank_team_position"] = (
        df.groupby(["team", "position"])["stat_MP"]
        .rank(method="dense", ascending=False)
    )

    df["starts_rank_team_position"] = (
        df.groupby(["team", "position"])["stat_SXI"]
        .rank(method="dense", ascending=False)
    )

    df["anytime_rank_team_position"] = (
        df.groupby(["team", "position"])["anytime_scorer_odd"]
        .rank(method="dense", ascending=True)
    )

    df["shots_rank_team_position"] = (
        df.groupby(["team", "position"])["stat_ST"]
        .rank(method="dense", ascending=False)
    )

    df["cc_rank_team_position"] = (
        df.groupby(["team", "position"])["stat_CC"]
        .rank(method="dense", ascending=False)
    )

    df["tackles_rank_team_position"] = (
        df.groupby(["team", "position"])["stat_T"]
        .rank(method="dense", ascending=False)
    )

    return df

In [ ]:
feature_functions = [
    minutes_engineered,
    goals_engineered,
    assists_engineered,
    clean_sheet_engineered,
    goals_conceded_engineered,
    midfield_engineered,
    forward_engineered,
    goalkeeper_engineered,
    discipline_engineered,
    penalties_engineered,
    form_engineered,
    relative_team_engineered,
]

for func in feature_functions:
    df = func(df)

In [ ]:
club_columns = []

for column in df.columns:
    if column.startswith("club_"):
        club_columns.append(column)

df = df.drop(columns=club_columns)


| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |
